## EDA

In [2]:
import pandas as pd

df = pd.read_csv("../DATA/PROCESSED/postsecondary_panel.csv.gz")
df.shape

(5662, 22)

In [3]:
#data types
print(df.dtypes)

unitid                                      int64
fall_year                                   int64
institution_name                           object
state                                      object
locale                                      int64
size_category                               int64
applicants                                  int64
admits                                    float64
enrolled                                  float64
sticker_tuition_fees                       object
average_net_price                         float64
first_time_percent_any_grant              float64
first_time_average_any_grant              float64
first_time_percent_institutional_grant    float64
first_time_average_institutional_grant    float64
first_time_total_institutional_grant      float64
tuition_fee_revenue                       float64
allowances_applied_to_tuition             float64
institutional_grants_funded               float64
institutional_grants_unfunded             float64


In [4]:

#number of unique values per column
df.nunique()

unitid                                    1211
fall_year                                    5
institution_name                          1277
state                                       52
locale                                      12
size_category                                6
applicants                                3724
admits                                    3186
enrolled                                  1345
sticker_tuition_fees                      3997
average_net_price                         4998
first_time_percent_any_grant                83
first_time_average_any_grant              5126
first_time_percent_institutional_grant     101
first_time_average_institutional_grant    4957
first_time_total_institutional_grant      5298
tuition_fee_revenue                       5520
allowances_applied_to_tuition             5377
institutional_grants_funded               4727
institutional_grants_unfunded             5104
allowances_endowment_gifts_tuition        3155
allowances_ot

In [8]:
# IPEDS uses "." for missing tuition, so replace it and convert to numbers
df = df.replace(".", pd.NA)
df["sticker_tuition_fees"] = pd.to_numeric(df["sticker_tuition_fees"])
print(df.isnull().sum())

unitid                                       0
fall_year                                    0
institution_name                             0
state                                        0
locale                                       0
size_category                                0
applicants                                   0
admits                                      38
enrolled                                    46
sticker_tuition_fees                        27
average_net_price                          192
first_time_percent_any_grant               150
first_time_average_any_grant               185
first_time_percent_institutional_grant     150
first_time_average_institutional_grant     318
first_time_total_institutional_grant       150
tuition_fee_revenue                        138
allowances_applied_to_tuition              138
institutional_grants_funded                138
institutional_grants_unfunded              138
allowances_endowment_gifts_tuition        1251
allowances_ot

In [10]:
by_year = df.groupby("fall_year").agg(
    entries=("unitid", "count"),
    missing=("allowances_endowment_gifts_tuition", lambda x: x.isna().sum())
)
by_year

,entries,missing
fall_year,,
2018,1142,1142
2019,1138,27
2020,1135,22
2021,1125,27
2022,1122,33


In [11]:
print(df["unitid"].nunique(), "institutions")

1211 institutions


In [13]:
# rows per year, and how many schools appear in all five years
print(df["fall_year"].value_counts().sort_index())
print((df.groupby("unitid").size() == 5).sum(), "schools in all five years")

fall_year
2018    1142
2019    1138
2020    1135
2021    1125
2022    1122
Name: count, dtype: int64
1059 schools in all five years


In [14]:
# states and territories outside the 50 states + DC
print(df["state"].value_counts()[["PR", "GU"]])

state
PR    113
GU      5
Name: count, dtype: int64


In [15]:
# quick check of the derived fields
yield_rate = df["enrolled"] / df["admits"]
discount = (df["first_time_percent_institutional_grant"] / 100
            * df["first_time_average_institutional_grant"]
            / df["sticker_tuition_fees"])

print((df["admits"] < 50).sum(), "rows with under 50 admits")
print((yield_rate == 1).sum(), "rows with 100% yield")
print((discount > 1).sum(), "rows with discount rate over 100%")

693 rows with under 50 admits
220 rows with 100% yield
19 rows with discount rate over 100%


In [12]:
df.duplicated(subset=["unitid", "fall_year"]).sum()

np.int64(0)

Summary: 
- 5662 rows, 22 columns. One row is one private nonprofit four-year college in one fall, 2018 to 2022.
- 1211 institutions. 1,122 to 1,142 per year. 1,059 appear in all five years.
- five IPEDS files per year (Directory, Admissions, Tuition, Financial Aid, Finance), joined on UNITID. Admissions is an inner join, so only schools that report admissions are included.
- 17 numeric, 3 categorical (state, locale, size), 1 identifier, 1 text. Tuition needed converting from text because of “.” values.
- no duplicates
- missing: low overall. Most columns are under 6%. The two allowance columns are 22% missing, all of 2018 plus about 2 to 3% in other years.
- target is yield (enrolled ÷ admits) isn’t a column yet. Ben creates it in EDA, along with admit rate and discount rate.
- next: drop the allowance columns or use 2019 on; minimum admits cutoff for tiny schools; keep or drop Puerto Rico and Guam; how to handle the 19 rows with a discount rate over 100%.